# Superconductivity（超导材料临界温度）

In [ ]:
from pathlib import Path

# 工具入参要求绝对路径；本 cell 算好后经 {VAR} 注入下方各命令
HERE = Path.cwd()
PY = str(HERE / "superconductor.py")      # :load / :load_X / :load_X_to_predict 按需拼后缀
YAML = str(HERE / "superconductor.yaml")


In [ ]:
# ① 数据画像（superconductor.py:load 全量数据）
!uv --project ../.. run --extra data mflowy data_profile '{"file_path": "{PY}:load"}'


In [ ]:
# ② 探索性数据分析（superconductor.py:load 全量数据）
!uv --project ../.. run --extra data mflowy eda '{"file_path": "{PY}:load", "target": "critical_temp"}'

In [ ]:
# ③ 多模型对比（从输出中记录 XGB 步骤的 run_id）
!uv --project ../.. run --extra model --extra deep mflowy modeling '{"modeling_steps_yaml": "{YAML}", "name": "super-v1", "desc": "Superconductor modeling"}'


In [ ]:
# ④ SHAP 解释（把下方 6b65a6066e214e94911f614ca16b7e31 替换为 ③ 输出的裸 run_id（无 XGB= 前缀））
!uv --project ../.. run --extra model --extra deep mflowy explanation '{"modeling_steps_yaml": "{YAML}", "model": "6b65a6066e214e94911f614ca16b7e31", "name": "super-shap", "desc": "Superconductor SHAP"}'


In [ ]:
# ⑤ 逆向搜索：目标 Tc 最大化（81 列搜索空间由 superconductor.py:load_X 自动推断，替代 X.csv）
!uv --project ../.. run --extra model --extra deep mflowy inverse_optimization '{"data": "{PY}:load_X", "model": "6b65a6066e214e94911f614ca16b7e31", "direction": {"critical_temp": "maximize"}, "n_trials": 200}'


In [ ]:
# ⑥ 上线前预测验证（superconductor.py:load_X_to_predict 取历史最高 Tc 样本；验证 top-1 候选需替换完整 81 列）
!uv --project ../.. run --extra model --extra deep mflowy predict '{"data": "{PY}:load_X_to_predict", "model": "6b65a6066e214e94911f614ca16b7e31"}'
